# JetBot Tape Line Follower (v3 — IPM + RANSAC)
Adds RANSAC polynomial fitting to the IPM bird's-eye path. Rejects outlier tape pixels (noise, shadows, stray edges) before fitting curves.
Stack: mask (Canny/Adaptive) -> morph close -> IPM warp -> sliding windows -> **RANSAC polyfit** -> EMA on poly coefficients -> midline -> PID w/ slew limit + miss grace.
Run cells top to bottom. Tune sliders while watching the live mask. Toggle **Enable Drive** when detection looks solid.

In [1]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import json
import time
from jetbot import Robot, Camera

In [2]:
camera = Camera.instance(width=300, height=300, fps=10)
robot = Robot()

In [4]:
# -- Stabilized all-in-one dashboard + IPM Polyfit + RANSAC (v3) --

# BEV canvas size (warped bird's-eye target). Kept fixed for consistent tuning.
BEV_W = 200
BEV_H = 400

# Camera feeds
cam_img  = widgets.Image(format='jpeg', width=300, height=300)
mask_img = widgets.Image(format='jpeg', width=300, height=300)
info_lbl = widgets.HTML(value='<i>Waiting for frames...</i>')

# -- Detection mode (mask method) --
mode = widgets.Dropdown(
    options=['Canny Edge', 'Adaptive Threshold'],
    value='Canny Edge', description='Mode')

# -- Canny sliders --
blur_k     = widgets.IntSlider(value=5,   min=1, max=21, step=2, description='Blur K')
canny_low  = widgets.IntSlider(value=50,  min=0, max=255, description='Canny Low')
canny_high = widgets.IntSlider(value=150, min=0, max=255, description='Canny High')
dilate_k   = widgets.IntSlider(value=3,   min=1, max=15, step=2, description='Dilate K')

# -- Adaptive threshold sliders --
block_size = widgets.IntSlider(value=11,  min=3,  max=51, step=2, description='Block Size')
c_offset   = widgets.IntSlider(value=2,   min=-20, max=20, description='C Offset')
invert     = widgets.ToggleButton(value=False, description='Invert',
                button_style='', layout=widgets.Layout(width='100px'))

# -- Mask cleanup --
close_k    = widgets.IntSlider(value=5,   min=1, max=15, step=2, description='Close K')

# -- Row-scan sliders (used when IPM disabled) --
min_run_w  = widgets.IntSlider(value=3,   min=1, max=30, description='Min Run W')
scan_rows  = widgets.IntSlider(value=12,  min=3, max=30, description='Scan Rows')
lane_w_px  = widgets.IntSlider(value=140, min=40, max=280, step=5, description='Lane Px (row)')

# -- ROI --
roi_pct    = widgets.FloatSlider(value=0.4,  min=0.1, max=1.0, step=0.05, description='ROI Bot %')

# -- IPM (bird's-eye warp) --
use_ipm      = widgets.Checkbox(value=False, description='Use IPM Polyfit')
trap_top_w   = widgets.FloatSlider(value=0.35, min=0.05, max=0.95, step=0.01, description='Trap Top W')
trap_top_y   = widgets.FloatSlider(value=0.10, min=0.0,  max=1.0,  step=0.01, description='Trap Top Y')
trap_bot_w   = widgets.FloatSlider(value=0.95, min=0.05, max=1.0,  step=0.01, description='Trap Bot W')
trap_bot_y   = widgets.FloatSlider(value=1.00, min=0.5,  max=1.0,  step=0.01, description='Trap Bot Y')
lane_w_bev   = widgets.IntSlider(value=100, min=20, max=BEV_W-10, step=2, description='Lane Px (BEV)')
n_windows    = widgets.IntSlider(value=9,   min=3, max=20, description='N Windows')
win_margin   = widgets.IntSlider(value=30,  min=5, max=100, description='Win Margin')
win_minpix   = widgets.IntSlider(value=20,  min=1, max=200, description='Win Min Px')
poly_ema     = widgets.FloatSlider(value=0.5, min=0.0, max=0.95, step=0.05, description='Poly EMA a')
show_bev     = widgets.Checkbox(value=True, description='Show BEV in mask pane')

# -- RANSAC sliders --
use_ransac        = widgets.Checkbox(value=True, description='Use RANSAC')
ransac_iters      = widgets.IntSlider(value=40,  min=5, max=200, description='RANSAC iters')
ransac_thresh     = widgets.FloatSlider(value=4.0, min=0.5, max=20.0, step=0.5, description='RANSAC thresh')
ransac_min_inlier = widgets.FloatSlider(value=0.35, min=0.1, max=0.9, step=0.05, description='Min inlier %')
ransac_degree     = widgets.Dropdown(options=[('Line (1)', 1), ('Quadratic (2)', 2)],
                                     value=2, description='Poly order')

# -- PID sliders --
kp         = widgets.FloatSlider(value=0.4,  min=0.0, max=2.0, step=0.01, description='Kp')
ki         = widgets.FloatSlider(value=0.0,  min=0.0, max=0.5, step=0.005, description='Ki')
kd         = widgets.FloatSlider(value=0.05, min=0.0, max=1.0, step=0.01, description='Kd')
err_ema    = widgets.FloatSlider(value=0.6,  min=0.0, max=0.95, step=0.05, description='Err EMA a')
integ_clip = widgets.FloatSlider(value=1.0,  min=0.1, max=5.0, step=0.1, description='I Clamp')

# -- Speed / drive sliders --
base_speed = widgets.FloatSlider(value=0.25, min=0.05, max=0.6, step=0.01, description='Base Speed')
curve_slow = widgets.FloatSlider(value=0.5,  min=0.0, max=1.0, step=0.05, description='Curve Slow')
lookahead  = widgets.FloatSlider(value=0.3,  min=0.0, max=1.0, step=0.05, description='Look-ahead')
min_wheel  = widgets.FloatSlider(value=0.0,  min=-0.5, max=0.15, step=0.01, description='Min Wheel')
slew_max   = widgets.FloatSlider(value=0.15, min=0.02, max=1.0, step=0.01, description='Slew/frame')
miss_grace = widgets.IntSlider(value=5,    min=0, max=20, description='Miss Grace')

# -- Drive toggle --
drive_enabled = widgets.ToggleButton(
    value=False, description='Enable Drive',
    button_style='danger', icon='car',
    layout=widgets.Layout(width='160px', height='40px'))

def on_drive_toggle(change):
    if change['new']:
        drive_enabled.button_style = 'success'
        drive_enabled.description = 'Driving!'
    else:
        drive_enabled.button_style = 'danger'
        drive_enabled.description = 'Enable Drive'
        robot.stop()
drive_enabled.observe(on_drive_toggle, names='value')

# -- Mode-specific widget groups --
canny_box    = widgets.VBox([blur_k, canny_low, canny_high, dilate_k])
adaptive_box = widgets.VBox([block_size, c_offset, invert])
adaptive_box.layout.display = 'none'
ransac_box   = widgets.VBox([use_ransac, ransac_degree, ransac_iters,
                             ransac_thresh, ransac_min_inlier])
ipm_box      = widgets.VBox([trap_top_w, trap_top_y, trap_bot_w, trap_bot_y,
                             lane_w_bev, n_windows, win_margin, win_minpix,
                             poly_ema, show_bev,
                             widgets.HTML('<b>RANSAC</b>'), ransac_box])
row_scan_box = widgets.VBox([scan_rows, lane_w_px])

def on_mode_change(change):
    if change['new'] == 'Canny Edge':
        canny_box.layout.display = ''
        adaptive_box.layout.display = 'none'
    else:
        canny_box.layout.display = 'none'
        adaptive_box.layout.display = ''
mode.observe(on_mode_change, names='value')

def on_ipm_toggle(change):
    if change['new']:
        ipm_box.layout.display = ''
        row_scan_box.layout.display = 'none'
    else:
        ipm_box.layout.display = 'none'
        row_scan_box.layout.display = ''
use_ipm.observe(on_ipm_toggle, names='value')
ipm_box.layout.display = 'none'

# -- Save / Load --
save_btn = widgets.Button(description='Save', button_style='info', icon='save')
load_btn = widgets.Button(description='Load', button_style='warning', icon='folder-open')
status_lbl = widgets.Label(value='')
CFG = 'line_follower_config_v3.json'

ALL_PARAMS = [('mode',mode),('blur_k',blur_k),('canny_low',canny_low),
              ('canny_high',canny_high),('dilate_k',dilate_k),
              ('block_size',block_size),('c_offset',c_offset),('invert',invert),
              ('close_k',close_k),('min_run_w',min_run_w),('roi_pct',roi_pct),
              ('scan_rows',scan_rows),('lane_w_px',lane_w_px),
              ('use_ipm',use_ipm),('trap_top_w',trap_top_w),('trap_top_y',trap_top_y),
              ('trap_bot_w',trap_bot_w),('trap_bot_y',trap_bot_y),
              ('lane_w_bev',lane_w_bev),('n_windows',n_windows),
              ('win_margin',win_margin),('win_minpix',win_minpix),
              ('poly_ema',poly_ema),('show_bev',show_bev),
              ('use_ransac',use_ransac),('ransac_iters',ransac_iters),
              ('ransac_thresh',ransac_thresh),('ransac_min_inlier',ransac_min_inlier),
              ('ransac_degree',ransac_degree),
              ('kp',kp),('ki',ki),('kd',kd),('err_ema',err_ema),('integ_clip',integ_clip),
              ('base_speed',base_speed),('curve_slow',curve_slow),
              ('lookahead',lookahead),('min_wheel',min_wheel),
              ('slew_max',slew_max),('miss_grace',miss_grace)]

def save_config(_):
    with open(CFG,'w') as f:
        json.dump({k: w.value for k,w in ALL_PARAMS}, f, indent=2)
    status_lbl.value = 'Saved!'
def load_config(_):
    try:
        with open(CFG) as f:
            cfg = json.load(f)
        for k, w in ALL_PARAMS:
            if k in cfg: w.value = cfg[k]
        status_lbl.value = 'Loaded!'
    except FileNotFoundError:
        status_lbl.value = 'No config file.'
save_btn.on_click(save_config)
load_btn.on_click(load_config)

# -- Layout --
left_col = widgets.VBox([
    widgets.HBox([cam_img, mask_img]),
    info_lbl
])

right_col = widgets.VBox([
    mode, use_ipm,
    widgets.HTML('<b>Mask</b>'),
    canny_box, adaptive_box, close_k, min_run_w, roi_pct,
    widgets.HTML('<hr style="margin:4px 0"><b>IPM / Polyfit</b>'),
    ipm_box,
    widgets.HTML('<hr style="margin:4px 0"><b>Row Scan</b>'),
    row_scan_box,
    widgets.HTML('<hr style="margin:4px 0"><b>PID / Steering</b>'),
    kp, ki, kd, err_ema, integ_clip,
    widgets.HTML('<hr style="margin:4px 0"><b>Speed / Drive</b>'),
    base_speed, curve_slow, lookahead, min_wheel, slew_max, miss_grace,
    drive_enabled,
    widgets.HBox([save_btn, load_btn, status_lbl]),
], layout=widgets.Layout(padding='0 0 0 12px'))

display(widgets.HBox([left_col, right_col]))

# -- Controller state --
_integral = 0.0
_prev_err_f = 0.0
_err_f = 0.0
_last_steer = 0.0
_last_lm = 0.0
_last_rm = 0.0
_miss_count = 0
_last_single_side = 0
_last_t = None
_poly_l_f = None    # smoothed left polynomial (2nd order)
_poly_r_f = None    # smoothed right polynomial


def get_runs(row):
    # Contiguous nonzero runs in a 1D binary row.
    padded = np.concatenate(([0], row, [0]))
    diff = np.diff(padded.astype(np.int16))
    starts = np.where(diff > 0)[0]
    ends   = np.where(diff < 0)[0]
    return list(zip(starts, ends))


# --------------------- Row-scan detection (non-IPM) ---------------------

def find_lane_points(mask, rw):
    global _last_single_side
    rh = mask.shape[0]
    n = scan_rows.value
    row_indices = np.linspace(rh - 1, 0, n, dtype=int)
    mrw = min_run_w.value
    half_lane = lane_w_px.value // 2
    points = []
    for r in row_indices:
        runs = get_runs(mask[r])
        runs = [(s, e) for s, e in runs if (e - s) >= mrw]
        if len(runs) >= 2:
            left_tape = runs[0]; right_tape = runs[-1]
            li = int(left_tape[1] - 1); ri = int(right_tape[0])
            cx = (li + ri) // 2
            points.append((int(r), li, ri, cx, 1.0))
        elif len(runs) == 1:
            s, e = runs[0]
            tape_center = (int(s) + int(e - 1)) // 2
            side = _last_single_side
            if side == 0:
                side = -1 if tape_center < rw // 2 else 1
            if side == -1:
                cx = tape_center + half_lane
            else:
                cx = tape_center - half_lane
            cx = int(np.clip(cx, 0, rw - 1))
            points.append((int(r), int(s), int(e - 1), cx, 0.5))
    return points


def update_single_side_hint(points, rw):
    global _last_single_side
    if not points: return
    twos = [p for p in points if p[4] >= 1.0]
    if twos:
        bottom = twos[0]; li, ri = bottom[1], bottom[2]
        if abs(li - rw // 2) < abs(ri - rw // 2):
            _last_single_side = -1
        else:
            _last_single_side = 1
    else:
        bottom = points[0]
        tape_center = (bottom[1] + bottom[2]) // 2
        _last_single_side = -1 if tape_center < rw // 2 else 1


def band_aggregate(points, h_roi, rw):
    if not points:
        return None, None, 0, 0
    near_list, far_list = [], []
    split_row = h_roi * 0.5
    for (r, li, ri, cx, _) in points:
        if r >= split_row: near_list.append(cx)
        else:              far_list.append(cx)
    near_cx = float(np.median(near_list)) if near_list else None
    far_cx  = float(np.median(far_list))  if far_list  else None
    return near_cx, far_cx, len(near_list), len(far_list)


# --------------------- IPM + polyfit detection ---------------------

def get_ipm_matrices(w, h):
    """Build forward + inverse perspective-transform matrices from current trap sliders."""
    tw = trap_top_w.value; ty = trap_top_y.value
    bw = trap_bot_w.value; by = trap_bot_y.value
    tl_x = (1.0 - tw) * 0.5 * w; tr_x = (1.0 + tw) * 0.5 * w
    bl_x = (1.0 - bw) * 0.5 * w; br_x = (1.0 + bw) * 0.5 * w
    top_y    = ty * h
    bottom_y = by * h
    src = np.float32([[tl_x, top_y], [tr_x, top_y],
                      [bl_x, bottom_y], [br_x, bottom_y]])
    dst = np.float32([[0, 0], [BEV_W, 0],
                      [0, BEV_H], [BEV_W, BEV_H]])
    M    = cv2.getPerspectiveTransform(src, dst)
    Minv = cv2.getPerspectiveTransform(dst, src)
    return M, Minv, src


def ransac_polyfit(ys, xs, degree, iters, thresh, min_inlier_ratio):
    """Minimal RANSAC polynomial fit. Returns (poly, inlier_mask) or (None, None).
    Samples (degree+1) points per iteration, counts pixels within `thresh` of the
    candidate curve, keeps the model with the most inliers, refits on the inlier set.
    """
    ys = np.asarray(ys); xs = np.asarray(xs)
    n = len(ys)
    ss = degree + 1
    if n < ss:
        return None, None
    best_inliers = None
    best_count = -1
    rng = np.random
    for _ in range(iters):
        idx = rng.choice(n, ss, replace=False)
        ys_s = ys[idx]; xs_s = xs[idx]
        if len(np.unique(ys_s)) < ss:
            continue  # degenerate (same y)
        try:
            cand = np.polyfit(ys_s, xs_s, degree)
        except (np.linalg.LinAlgError, ValueError, TypeError):
            continue
        resid = np.abs(np.polyval(cand, ys) - xs)
        inliers = resid < thresh
        c = int(inliers.sum())
        if c > best_count:
            best_count = c
            best_inliers = inliers
    if best_inliers is None:
        return None, None
    if best_count < max(ss, int(n * min_inlier_ratio)):
        return None, None
    try:
        refit = np.polyfit(ys[best_inliers], xs[best_inliers], degree)
    except (np.linalg.LinAlgError, ValueError, TypeError):
        return None, None
    return refit, best_inliers


def _fit_points(ys, xs, fallback_degree):
    """Fit polynomial via RANSAC or plain polyfit, respecting current slider state.
    Returns (poly, inlier_mask_or_None).
    """
    if len(ys) < fallback_degree + 1:
        return None, None
    if use_ransac.value:
        deg = int(ransac_degree.value)
        poly, inliers = ransac_polyfit(ys, xs, deg,
                                       iters=ransac_iters.value,
                                       thresh=ransac_thresh.value,
                                       min_inlier_ratio=ransac_min_inlier.value)
        if poly is not None:
            # Pad to quadratic shape for EMA compatibility
            if deg < 2:
                poly = np.concatenate([np.zeros(2 - deg), poly])
            return poly, inliers
        # fall through: plain polyfit as rescue
    try:
        poly = np.polyfit(ys, xs, 2)
        return poly, None
    except (np.linalg.LinAlgError, ValueError, TypeError):
        return None, None


def sliding_window_fit(warped):
    """Histogram bottom, sliding window up each side, 2nd-order polyfit (x = a*y^2 + b*y + c).
    Returns (poly_l, poly_r, dbg) where dbg is a visualization BGR image.
    """
    H, W = warped.shape
    bev_bgr = cv2.cvtColor(warped, cv2.COLOR_GRAY2BGR)

    hist = np.sum(warped[H // 2:, :], axis=0)
    mid = W // 2
    left_peak  = int(np.argmax(hist[:mid])) if hist[:mid].max() > 0 else None
    right_peak = int(mid + np.argmax(hist[mid:])) if hist[mid:].max() > 0 else None

    nw = n_windows.value
    margin = win_margin.value
    minpix = win_minpix.value
    wh = H // nw

    def slide(x_start):
        if x_start is None: return None, 0, 0
        x = x_start
        ys_all, xs_all = [], []
        for wi in range(nw):
            y_lo = H - (wi + 1) * wh
            y_hi = H - wi * wh
            x_lo = max(0, x - margin); x_hi = min(W, x + margin)
            cv2.rectangle(bev_bgr, (x_lo, y_lo), (x_hi, y_hi), (0, 180, 0), 1)
            win = warped[y_lo:y_hi, x_lo:x_hi]
            ys, xs = np.nonzero(win)
            if len(xs) > minpix:
                xs_g = xs + x_lo; ys_g = ys + y_lo
                ys_all.append(ys_g); xs_all.append(xs_g)
                x = int(np.mean(xs_g))
        if ys_all:
            ys_cat = np.concatenate(ys_all); xs_cat = np.concatenate(xs_all)
            if len(ys_cat) > 10:
                poly, inliers = _fit_points(ys_cat, xs_cat, fallback_degree=2)
                if poly is not None and inliers is not None:
                    in_mask = inliers
                    out_mask = ~in_mask
                    ys_in = ys_cat[in_mask]; xs_in = xs_cat[in_mask]
                    ys_out = ys_cat[out_mask]; xs_out = xs_cat[out_mask]
                    bev_bgr[ys_in, xs_in] = (0, 255, 0)
                    if len(ys_out):
                        bev_bgr[ys_out, xs_out] = (0, 0, 255)
                    return poly, int(in_mask.sum()), int(out_mask.sum())
                return poly, len(ys_cat), 0
        return None, 0, 0

    poly_l, inl_l, out_l = slide(left_peak)
    poly_r, inl_r, out_r = slide(right_peak)
    return poly_l, poly_r, bev_bgr, (inl_l, out_l, inl_r, out_r)


def reconcile_polys(poly_l, poly_r):
    """Fallback: if only one poly, derive the other by offsetting a*y+b+c by +/- lane_w_bev."""
    lw = lane_w_bev.value
    if poly_l is not None and poly_r is None:
        poly_r = poly_l.copy(); poly_r[-1] = poly_l[-1] + lw
    elif poly_r is not None and poly_l is None:
        poly_l = poly_r.copy(); poly_l[-1] = poly_r[-1] - lw
    return poly_l, poly_r


def ema_poly(prev, new, a):
    if new is None: return prev
    if prev is None: return new
    if prev.shape != new.shape: return new
    return a * prev + (1.0 - a) * new


# --------------------- Frame processor ---------------------

def process_frame(change):
    global _integral, _prev_err_f, _err_f, _last_steer
    global _last_lm, _last_rm, _miss_count, _last_t
    global _poly_l_f, _poly_r_f

    now = time.time()
    dt = 0.1 if _last_t is None else max(1e-3, min(0.5, now - _last_t))
    _last_t = now

    frame = change['new']
    h, w = frame.shape[:2]

    roi_top = int(h * (1.0 - roi_pct.value))
    roi = frame[roi_top:, :]
    h_roi = roi.shape[0]
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

    bk = blur_k.value | 1
    blurred = cv2.GaussianBlur(gray, (bk, bk), 0)

    if mode.value == 'Canny Edge':
        edges = cv2.Canny(blurred, canny_low.value, canny_high.value)
        dk = dilate_k.value | 1
        mask = cv2.dilate(edges, np.ones((dk, dk), np.uint8), iterations=2)
    else:
        bs = block_size.value | 1
        if bs < 3: bs = 3
        thresh_type = cv2.THRESH_BINARY_INV if not invert.value else cv2.THRESH_BINARY
        mask = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     thresh_type, bs, c_offset.value)
        kern = np.ones((3, 3), np.uint8)
        mask = cv2.erode(mask, kern, iterations=1)
        mask = cv2.dilate(mask, kern, iterations=2)

    ck = close_k.value | 1
    if ck >= 3:
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE,
                                np.ones((ck, ck), np.uint8))

    ov = frame.copy()
    cv2.line(ov, (0, roi_top), (w, roi_top), (0, 0, 255), 2)
    cv2.line(ov, (w // 2, roi_top), (w // 2, h), (128, 128, 128), 1)

    # ==================== IPM PATH ====================
    if use_ipm.value:
        # Mask is in ROI coords; warp from full frame coords so trap sliders act on full image.
        # Build mask in full-frame coordinates by embedding ROI mask into a full-size mask.
        full_mask = np.zeros((h, w), np.uint8)
        full_mask[roi_top:, :] = mask

        M, Minv, src_pts = get_ipm_matrices(w, h)

        # Draw trapezoid overlay on camera view
        pts = src_pts.astype(int)
        cv2.polylines(ov, [pts.reshape(-1, 1, 2)], True, (255, 200, 0), 2)

        warped = cv2.warpPerspective(full_mask, M, (BEV_W, BEV_H),
                                     flags=cv2.INTER_NEAREST)
        poly_l, poly_r, bev_dbg, ransac_stats = sliding_window_fit(warped)
        poly_l, poly_r = reconcile_polys(poly_l, poly_r)

        # EMA smoothing of polys
        a_p = poly_ema.value
        _poly_l_f = ema_poly(_poly_l_f, poly_l, a_p)
        _poly_r_f = ema_poly(_poly_r_f, poly_r, a_p)

        if _poly_l_f is None or _poly_r_f is None:
            _miss_count += 1
            _no_line(ov, h)
            mask_disp = bev_dbg if show_bev.value else cv2.cvtColor(full_mask, cv2.COLOR_GRAY2BGR)
        else:
            poly_mid = (_poly_l_f + _poly_r_f) / 2.0

            # Error at two y positions: near (bottom of BEV) and far (upper third)
            y_near = BEV_H - 1
            y_far  = BEV_H // 4
            x_near = np.polyval(poly_mid, y_near)
            x_far  = np.polyval(poly_mid, y_far)

            # Normalize relative to BEV center
            near_err = (x_near - BEV_W / 2) / (BEV_W / 2)
            far_err  = (x_far  - BEV_W / 2) / (BEV_W / 2)
            la = lookahead.value
            error = (1.0 - la) * near_err + la * far_err

            # Curvature (signed) from 2nd-order coefficient of midline
            # d2x/dy2 = 2*a -> scale for display only
            curv = float(2.0 * poly_mid[0])

            # Draw fitted lines on BEV dbg
            ys_draw = np.linspace(0, BEV_H - 1, 30)
            xs_l = np.polyval(_poly_l_f, ys_draw)
            xs_r = np.polyval(_poly_r_f, ys_draw)
            xs_m = np.polyval(poly_mid,  ys_draw)
            for (xs, col) in ((xs_l, (0, 0, 255)), (xs_r, (255, 0, 0)), (xs_m, (0, 255, 0))):
                pts_draw = np.stack([xs, ys_draw], axis=1).astype(int)
                cv2.polylines(bev_dbg, [pts_draw.reshape(-1, 1, 2)], False, col, 2)

            # Project midline back into camera view
            ys_cam = np.linspace(BEV_H - 1, 0, 20)
            xs_cam = np.polyval(poly_mid, ys_cam)
            bev_pts = np.stack([xs_cam, ys_cam, np.ones_like(ys_cam)], axis=1).T  # 3xN
            cam_pts_h = Minv @ bev_pts
            cam_pts = (cam_pts_h[:2] / cam_pts_h[2]).T.astype(int)
            cv2.polylines(ov, [cam_pts.reshape(-1, 1, 2)], False, (0, 255, 0), 2)

            # PID (same as row-scan path)
            _err_f = err_ema.value * _err_f + (1.0 - err_ema.value) * error
            deriv = (_err_f - _prev_err_f) / dt
            leak = 0.97
            _integral = _integral * leak + _err_f * dt
            _integral = float(np.clip(_integral, -integ_clip.value, integ_clip.value))
            steer_raw = (kp.value * _err_f + ki.value * _integral + kd.value * deriv)
            steer = float(np.clip(steer_raw, -1.0, 1.0))
            if abs(steer_raw) > 1.0 and np.sign(steer_raw) == np.sign(_err_f):
                _integral -= _err_f * dt
            _prev_err_f = _err_f

            speed = base_speed.value * (1.0 - curve_slow.value * abs(steer))
            lm = speed + steer; rm = speed - steer
            mw = min_wheel.value
            lm = float(np.clip(lm, mw, 1.0)); rm = float(np.clip(rm, mw, 1.0))
            sm = slew_max.value
            lm = float(np.clip(lm, _last_lm - sm, _last_lm + sm))
            rm = float(np.clip(rm, _last_rm - sm, _last_rm + sm))

            if drive_enabled.value:
                robot.set_motors(lm, rm)
                _last_lm, _last_rm = lm, rm
            else:
                _last_lm, _last_rm = 0.0, 0.0

            _last_steer = steer
            _miss_count = 0

            inl_l, out_l, inl_r, out_r = ransac_stats
            mode_tag = 'IPM+RANSAC' if use_ransac.value else 'IPM'
            info_lbl.value = (f'<b style="color:green">TRACK/{mode_tag}</b> &nbsp; '
                             f'err:{error:+.2f} ef:{_err_f:+.2f} &nbsp; '
                             f'steer:{steer:+.2f} curv:{curv:+.4f} &nbsp; '
                             f'spd:{speed:.2f} L:{lm:.2f} R:{rm:.2f} &nbsp; '
                             f'inL:{inl_l}/{inl_l+out_l} inR:{inl_r}/{inl_r+out_r} &nbsp; '
                             f'dt:{dt*1000:.0f}ms')
            mask_disp = bev_dbg if show_bev.value else cv2.cvtColor(full_mask, cv2.COLOR_GRAY2BGR)

        _, j1 = cv2.imencode('.jpg', ov)
        cam_img.value = j1.tobytes()
        mask_disp_resized = cv2.resize(mask_disp, (300, 300)) if mask_disp.shape[:2] != (300, 300) else mask_disp
        _, j2 = cv2.imencode('.jpg', mask_disp_resized)
        mask_img.value = j2.tobytes()
        return

    # ==================== ROW-SCAN PATH ====================
    points = find_lane_points(mask, w)

    if points:
        update_single_side_hint(points, w)
        for r, lx, rx, cx, conf in points:
            yr = r + roi_top
            col_edge = (0, 0, 255) if conf >= 1.0 else (0, 180, 180)
            cv2.circle(ov, (lx, yr), 4, col_edge, -1)
            cv2.circle(ov, (rx, yr), 4, col_edge, -1)
            cv2.circle(ov, (cx, yr), 5, (0, 255, 0), -1)

        near_cx, far_cx, near_n, far_n = band_aggregate(points, h_roi, w)

        if near_cx is None and far_cx is None:
            _miss_count += 1
            _no_line(ov, h)
        else:
            if near_cx is None: near_cx = far_cx
            if far_cx  is None: far_cx  = near_cx
            near_err = (near_cx - w / 2) / (w / 2)
            far_err  = (far_cx  - w / 2) / (w / 2)
            la = lookahead.value * (1.0 if far_n >= 2 else 0.3)
            error = (1.0 - la) * near_err + la * far_err
            _err_f = err_ema.value * _err_f + (1.0 - err_ema.value) * error
            deriv = (_err_f - _prev_err_f) / dt
            leak = 0.97
            _integral = _integral * leak + _err_f * dt
            _integral = float(np.clip(_integral, -integ_clip.value, integ_clip.value))
            steer_raw = (kp.value * _err_f + ki.value * _integral + kd.value * deriv)
            steer = float(np.clip(steer_raw, -1.0, 1.0))
            if abs(steer_raw) > 1.0 and np.sign(steer_raw) == np.sign(_err_f):
                _integral -= _err_f * dt
            _prev_err_f = _err_f
            speed = base_speed.value * (1.0 - curve_slow.value * abs(steer))
            lm = speed + steer; rm = speed - steer
            mw = min_wheel.value
            lm = float(np.clip(lm, mw, 1.0)); rm = float(np.clip(rm, mw, 1.0))
            sm = slew_max.value
            lm = float(np.clip(lm, _last_lm - sm, _last_lm + sm))
            rm = float(np.clip(rm, _last_rm - sm, _last_rm + sm))
            if drive_enabled.value:
                robot.set_motors(lm, rm)
                _last_lm, _last_rm = lm, rm
            else:
                _last_lm, _last_rm = 0.0, 0.0
            _last_steer = steer; _miss_count = 0

            near_y = (points[0][0] if points else h_roi - 1) + roi_top
            far_y  = (points[-1][0] if points else 0) + roi_top
            cv2.circle(ov, (int(near_cx), near_y), 10, (0, 255, 255), -1)
            cv2.circle(ov, (int(far_cx), far_y), 8, (255, 255, 0), 2)
            blend_x = int((1.0 - la) * near_cx + la * far_cx)
            cv2.line(ov, (w // 2, near_y), (blend_x, near_y), (255, 0, 255), 2)
            conf_avg = np.mean([p[4] for p in points])
            info_lbl.value = (f'<b style="color:green">TRACK/ROW</b> &nbsp; '
                             f'err:{error:+.2f} ef:{_err_f:+.2f} &nbsp; '
                             f'steer:{steer:+.2f} spd:{speed:.2f} &nbsp; '
                             f'L:{lm:.2f} R:{rm:.2f} &nbsp; '
                             f'conf:{conf_avg:.2f} near:{near_n} far:{far_n} dt:{dt*1000:.0f}ms')
    else:
        _miss_count += 1
        _no_line(ov, h)

    _, j1 = cv2.imencode('.jpg', ov)
    cam_img.value = j1.tobytes()
    mask_full = np.zeros((h, w), np.uint8)
    mask_full[roi_top:, :] = mask
    _, j2 = cv2.imencode('.jpg', cv2.cvtColor(mask_full, cv2.COLOR_GRAY2BGR))
    mask_img.value = j2.tobytes()


def _no_line(ov, h):
    global _integral, _prev_err_f, _err_f, _last_lm, _last_rm, _poly_l_f, _poly_r_f
    grace = miss_grace.value
    if _miss_count <= grace and drive_enabled.value:
        decay = 0.6
        lm = _last_lm * decay; rm = _last_rm * decay
        sm = slew_max.value
        lm = float(np.clip(lm, _last_lm - sm, _last_lm + sm))
        rm = float(np.clip(rm, _last_rm - sm, _last_rm + sm))
        robot.set_motors(lm, rm)
        _last_lm, _last_rm = lm, rm
        cv2.putText(ov, f'HOLD {_miss_count}/{grace}', (10, h // 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 180, 255), 2)
        info_lbl.value = (f'<b style="color:orange">HOLDING</b> '
                         f'{_miss_count}/{grace} - L:{lm:.2f} R:{rm:.2f}')
    else:
        _integral = 0.0; _prev_err_f = 0.0; _err_f = 0.0
        _last_lm = 0.0; _last_rm = 0.0
        _poly_l_f = None; _poly_r_f = None
        if drive_enabled.value:
            robot.stop()
        cv2.putText(ov, 'LINE LOST', (10, h // 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        info_lbl.value = '<b style="color:red">LINE LOST</b> - motors stopped'


camera.observe(process_frame, names='value')
print('v3 dashboard active. Toggle "Use IPM Polyfit" + "Use RANSAC" for robust curve fitting.')

v3 dashboard active. Toggle "Use IPM Polyfit" + "Use RANSAC" for robust curve fitting.


In [ ]:
# -- STOP -- run this cell to halt everything
camera.unobserve_all()
robot.stop()
drive_enabled.value = False
print('Stopped.')